---
## Task 2 — Shots: High-Possession Teams vs. Low-Possession Teams

### Analytic question formulation
Do teams that hold **more than 50% possession** in a match register significantly more
**shots** than teams that hold **less than 50% possession**?
(Possession share is used here as a proxy for territorial control, and we test whether it
translates into a higher shot volume.)

**H0:** mean shots(possession > 50%) = mean shots(possession < 50%)
**H1:** mean shots(possession > 50%) > mean shots(possession < 50%)

### Data wrangling
Load the raw match-level CSV and reshape it from one row per match (home/away columns) into
one row per **team-match** observation, keeping each team's possession share and shot count.

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import scipy.stats as st

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your CSV file
file_path = '/content/drive/MyDrive/Colab/world_cup_match_data.csv'

df = pd.read_csv(file_path)
print("CSV imported successfully!")
print(df.head())

Mounted at /content/drive
CSV imported successfully!
    timestamp              date_GMT    status  attendance home_team_name  \
0  1781204400  Jun 11 2026 - 7:00pm  complete       80824         Mexico   
1  1781229600  Jun 12 2026 - 2:00am  complete       44985    South Korea   
2  1781290800  Jun 12 2026 - 7:00pm  complete       43002         Canada   
3  1781312400  Jun 13 2026 - 1:00am  complete       70492          USMNT   
4  1781377200  Jun 13 2026 - 7:00pm  complete       67966          Qatar   

           away_team_name  referee  Game Week  Pre-Match PPG (Home)  \
0            South Africa      NaN        1.0                   0.0   
1          Czech Republic      NaN        1.0                   0.0   
2  Bosnia and Herzegovina      NaN        1.0                   0.0   
3                Paraguay      NaN        1.0                   0.0   
4             Switzerland      NaN        1.0                   0.0   

   Pre-Match PPG (Away)  ...  odds_ft_home_team_win  odds_ft_dr

In [2]:
df2 = df[
        [
            'home_team_name',
            'away_team_name',
            'home_team_shots',
            'away_team_shots',
            'home_team_possession',
            'away_team_possession'
        ]
    ].copy()
df2['match'] = df2['home_team_name'] + ' vs ' + df2['away_team_name']

df_long = pd.concat([
    df2[['match', 'home_team_name', 'home_team_possession', 'home_team_shots']]
        .rename(columns={
            'home_team_name': 'team',
            'home_team_possession': 'possession',
            'home_team_shots': 'shots'
        }),

    df2[['match', 'away_team_name', 'away_team_possession', 'away_team_shots']]
        .rename(columns={
            'away_team_name': 'team',
            'away_team_possession': 'possession',
            'away_team_shots': 'shots'
        })
], ignore_index=True)

df_long.head()

,match,team,possession,shots
0,Mexico vs South Africa,Mexico,61,16
1,South Korea vs Czech Republic,South Korea,62,15
2,Canada vs Bosnia and Herzegovina,Canada,61,13
3,USMNT vs Paraguay,USMNT,65,16
4,Qatar vs Switzerland,Qatar,32,6


### Data preparation and sampling
**Population:** every team-match observation, split into a high-possession group
(possession > 50%) and a low-possession group (possession < 50%); observations at exactly
50% possession are excluded since they belong to neither group.
**Sample:** an independent **simple random sample of n = 30** drawn separately from the
high-possession and low-possession groups, for a two-sample comparison.

In [3]:
high_possession = df_long[df_long['possession'] > 50]['shots']
low_possession  = df_long[df_long['possession'] < 50]['shots']

print("Population size — high possession:", len(high_possession))
print("Population size — low possession:", len(low_possession))

n = 30
high_sample = high_possession.sample(n=n, random_state=7).reset_index(drop=True)
low_sample  = low_possession.sample(n=n, random_state=7).reset_index(drop=True)

print("High-possession sample n:", len(high_sample), " Low-possession sample n:", len(low_sample))

Population size — high possession: 104
Population size — low possession: 104
High-possession sample n: 30  Low-possession sample n: 30


### Descriptive statistics

In [4]:
for name, s in [("High possession (>50%)", high_sample), ("Low possession (<50%)", low_sample)]:
    print(f"\n{name}:")
    print(s.describe())


High possession (>50%):
count    30.000000
mean     14.866667
std       6.946760
min       5.000000
25%       8.500000
50%      15.000000
75%      19.000000
max      32.000000
Name: shots, dtype: float64

Low possession (<50%):
count    30.000000
mean      8.400000
std       4.846613
min       2.000000
25%       5.000000
50%       7.000000
75%      10.750000
max      22.000000
Name: shots, dtype: float64


### Inferential statistics — Confidence intervals (95%, each group)

In [5]:
for name, s in [("High possession (>50%)", high_sample), ("Low possession (<50%)", low_sample)]:
    m, sem = s.mean(), st.sem(s)
    ci = st.t.interval(0.95, df=len(s) - 1, loc=m, scale=sem)
    print(f"{name}: mean = {m:.3f}, 95% CI = ({ci[0]:.3f}, {ci[1]:.3f})")

High possession (>50%): mean = 14.867, 95% CI = (12.273, 17.461)
Low possession (<50%): mean = 8.400, 95% CI = (6.590, 10.210)


### Inferential statistics — Two-sample t-Test (Welch's, one-tailed)
H0: μ(high possession) = μ(low possession)  vs.  H1: μ(high possession) > μ(low possession)

In [6]:
t_stat, p_val = st.ttest_ind(high_sample, low_sample, equal_var=False, alternative='greater')
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")

alpha = 0.05
conclusion = "Reject H0" if p_val < alpha else "Fail to reject H0"
print("Conclusion:", conclusion, "at the 5% significance level.")

t-statistic = 4.182, p-value = 0.0001
Conclusion: Reject H0 at the 5% significance level.


In [7]:
# diff = high_sample.mean() - low_sample.mean()
# sig = "statistically significant" if p_val < alpha else "not statistically significant"
# print(
#     f"Interpretation: In the sample, high-possession teams averaged {high_sample.mean():.2f} shots "
#     f"vs {low_sample.mean():.2f} for low-possession teams (difference = {diff:.2f}). "
#     f"With p = {p_val:.4f}, this result is {sig} at the 5% level, so we {conclusion.lower()} "
#     f"that higher possession is associated with significantly more shots."
# )

Interpretation: In the sample, high-possession teams averaged 14.87 shots vs 8.40 for low-possession teams (difference = 6.47). With p = 0.0001, this result is statistically significant at the 5% level, so we reject h0 that higher possession is associated with significantly more shots.


**Interpretation:** high-possession teams averaged 14.87 shots vs 8.40 for low-possession teams, higher possession is associated with significantly more shots